In [1]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet201
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam

#!pip install tensorflow
from tensorflow.keras.applications import DenseNet201
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam

base_model = DenseNet201(
    weights='imagenet',
    include_top=False,
    input_shape=(128,128,3)
)


base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),

    Dense(512, activation='relu'),
    Dropout(0.5),

    Dense(256, activation='relu'),
    Dropout(0.5),

    Dense(7, activation='softmax')
])



model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


74836368/74836368 ━━━━━━━━━━━━━━━━━━━━ 10s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet201 (Functional)        │ (None, 4, 4, 1920)     │    18,321,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1920)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       983,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,438,663 (74.15 MB)

 Trainable params: 1,116,679 (4.26 MB)

 Non-trainable params: 18,321,984 (69.89 MB)

In [2]:
model.load_weights('final_tea_model2.h5')

In [3]:
import os
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# Load your trained model (if not already loaded)
# model = load_model('best_tea_model.h5')  # Uncomment if model not in memory

img_size = 128

# Load and preprocess single image
imgPath = os.path.join('image.png')

imgArr = cv2.imread(imgPath)
imgArr = cv2.cvtColor(imgArr, cv2.COLOR_BGR2RGB)
imgArr = cv2.resize(imgArr, (img_size, img_size))
imgArr = imgArr.astype("float32") / 255.0

# CRITICAL FIX: Add batch dimension
imgArr = np.expand_dims(imgArr, axis=0)  # Shape becomes (1, 128, 128, 3)

# Predict
result = model.predict(imgArr)
label = np.argmax(result)
confidence = np.max(result)

# Class names (update these to match your actual classes)
class_names = [
    'Tea algal leaf spot',
    'Brown Blight', 
    'Gray Blight',
    'Helopeltis',
    'Red spider',
    'Green mirid bug',
    'Healthy leaf'
]

print(f"Predicted Class Index: {label}")
print(f"Predicted Disease: {class_names[label]}")
print(f"Confidence: {confidence:.2%} ({confidence*100:.2f}%)")

# Optional: Show all class probabilities
print("\nAll Class Probabilities:")
for i, name in enumerate(class_names):
    print(f"  {name:25s}: {result[0][i]:.2%}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step
Predicted Class Index: 1
Predicted Disease: Brown Blight
Confidence: 98.70% (98.70%)

All Class Probabilities:
  Tea algal leaf spot      : 0.07%
  Brown Blight             : 98.70%
  Gray Blight              : 0.95%
  Helopeltis               : 0.01%
  Red spider               : 0.24%
  Green mirid bug          : 0.04%
  Healthy leaf             : 0.00%


In [7]:
print(result)

[[0.14291964 0.14241718 0.14356351 0.1415666  0.14321591 0.14411321
  0.14220393]]
